In [11]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

In [12]:
df = pd.read_csv("/Users/ameypatil/Desktop/Tata Power/data/final.csv")

In [13]:
# Drop unwanted columns
df = df.drop(columns=['_id_x', '_id_y'])

# Datetime handling
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

# Time-of-day feature
df['hour'] = df['datetime'].dt.hour

# Drop NaN
df = df.dropna()

In [14]:
Y = df['wdLoad'].values

In [15]:
T = df['temp'].values
X = df[['hour','rh']].values

In [16]:
kf = KFold(n_splits=2, shuffle=True, random_state=42)

Y_res = np.zeros_like(Y)
T_res = np.zeros_like(T)

for train_idx, test_idx in kf.split(X):
    
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    
    # Nuisance models
    model_y = RandomForestRegressor()
    model_t = RandomForestRegressor()
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    # Residuals
    Y_res[test_idx] = Y_test - model_y.predict(X_test)
    T_res[test_idx] = T_test - model_t.predict(X_test)

# Final regression
final_model = LinearRegression()
final_model.fit(T_res.reshape(-1,1), Y_res)

tau_temp = final_model.coef_[0]

print("ATE (Temp - DML):", tau_temp)

ATE (Temp - DML): 8.726146585264198


In [17]:
T = df['rh'].values
X = df[['hour','temp']].values

In [18]:
Y_res = np.zeros_like(Y)
T_res = np.zeros_like(T)

for train_idx, test_idx in kf.split(X):
    
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    
    model_y = RandomForestRegressor()
    model_t = RandomForestRegressor()
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_res[test_idx] = Y_test - model_y.predict(X_test)
    T_res[test_idx] = T_test - model_t.predict(X_test)

final_model = LinearRegression()
final_model.fit(T_res.reshape(-1,1), Y_res)

tau_rh = final_model.coef_[0]

print("ATE (Humidity - DML):", tau_rh)

ATE (Humidity - DML): 0.5443817558596946


In [19]:
T = df[['temp','rh']].values
X = df[['hour']].values

In [20]:
Y_res = np.zeros_like(Y)
T_res = np.zeros_like(T)

for train_idx, test_idx in kf.split(X):
    
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    
    model_y = RandomForestRegressor()
    model_t = RandomForestRegressor()
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_res[test_idx] = Y_test - model_y.predict(X_test)
    T_res[test_idx] = T_test - model_t.predict(X_test)

In [21]:
final_model = LinearRegression()
final_model.fit(T_res, Y_res)

tau_multi = final_model.coef_

print("ATE (Temp):", tau_multi[0])
print("ATE (Humidity):", tau_multi[1])

ATE (Temp): 7.621218350912563
ATE (Humidity): 0.40464996908199863


In [22]:
print(df[['temp','wdLoad']].corr())

            temp    wdLoad
temp    1.000000  0.507902
wdLoad  0.507902  1.000000


In [23]:
print(df[['rh','wdLoad']].corr())

              rh    wdLoad
rh      1.000000 -0.235082
wdLoad -0.235082  1.000000
